In [1]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype, is_bool_dtype

### Tree Classes
The tree will be built on "node" and "leaf" classes. Each node will have children, which aim to further purify the tree. Leafs are the end result, if an unseen data point reaches a leaf, the unseen datapoint is then classified to the "output_value"'s stored value.

In [3]:
class node:
    def __init__(self, data, splitting_attribute, branch_condition, parent):
        self.data = data
        self.parent = parent
        self.splitting_attribute = splitting_attribute
        self.branch_condition = branch_condition
        self.children = {}
    
    def partitioning(self):
        feature = self.splitting_attribute
 
        if is_bool_dtype(self.data[feature]) or isinstance(self.data[feature].dtype, pd.CategoricalDtype):
            for category in self.data[feature].dropna().unique():
                self.children[f'child_{len(self.children)}'] = leaf(self.data[self.data[feature] == category].drop(columns=self.splitting_attribute, axis=1), 
                                                                    lambda row, col=feature, cat=category: row[col] == cat,
                                                                    self
                                                                   )
                
        elif is_numeric_dtype(self.data[feature]):
            self.data = self.data.sort_values(by=feature, ascending=True).reset_index(drop=True)

            values = self.data[feature].values
            
            lowest_gini_val = float('inf')
            best_children = None

            for i in range(len(values) - 1):
                avg_adjacent_value = (values[i] + values[i+1]) / 2

                best_children = {
                    f'child_0': leaf(self.data[self.data[feature] < avg_adjacent_value], 
                                                                                  lambda row, col=feature, avg=avg_adjacent_value: row[col] < avg, 
                                                                                  self),
                    f'child_1': leaf(self.data[self.data[feature] >= avg_adjacent_value], 
                                                                                  lambda row, col=feature, avg=avg_adjacent_value: row[col] >= avg, 
                                                                                  self)
                }
                
                curr_gini_val = self.calculate_total_gini_impurity(best_children)
                    
                if curr_gini_val < lowest_gini_val:
                        lowest_gini_val = curr_gini_val
                        self.children = best_children
            
    def calculate_total_gini_impurity(self, children: dict) -> float:
        total_gini_impurity = 0.0
        
        for _, value in children.items():
            total_gini_impurity += (len(value.get_data()) / len(self.data)) * value.calculate_gini_impurity()
            
        return total_gini_impurity

    def get_data(self):
        return self.data

    def get_parent(self):
        return self.parent
        
    def get_children(self):
        return self.children

    def get_branch_condition(self):
        return self.branch_condition

In [4]:
class leaf:
    def __init__(self, data, branch_condition, parent):
        self.data = data
        self.parent = parent
        self.branch_condition = branch_condition
        self.output_value = None
    
    def calculate_gini_impurity(self) -> float:
        counts = self.data[target].value_counts()
        probs = counts / len(self.data)
        return 1 - (probs ** 2).sum()

    def set_output_value(self):
        self.output_value = self.data[target].mode().iloc[0]

    def get_output_value(self):
        return self.output_value
        
    def get_data(self):
        return self.data
        
    def get_parent(self):
        return self.parent

    def get_branch_condition(self):
        return self.branch_condition

### Tree Building
The tree's functionality is mostly present in the code below. The partition method performs the tree's purification, growing the tree new nodes and leafs to better sub-divide each possible class, an unseen data point could be classified to.

In [6]:
def add_to_tree(element_to_add):
    ##---Add Element To Tree--##
    tree[f'partition_{len(tree)}'] = element_to_add

In [7]:
def partition():
    
    for key, element in tree.copy().items():
        ##---Skip Nodes---##
        if not isinstance(element, leaf):
            continue;

        ##---Skip Pure Leafs---##
        if element.calculate_gini_impurity() != 0.0:

            ##---Create New Node---##
            lowest_gini_val = float('inf')
            best_node = None
            
            for predictor in predictors:
                if predictor not in element.get_data().columns:
                    continue;
                    
                curr_node = node(element.get_data(), predictor, element.get_branch_condition(), element.get_parent())
                curr_node.partitioning()
                curr_gini_val = curr_node.calculate_total_gini_impurity(curr_node.get_children())
        
                if curr_gini_val < lowest_gini_val:
                    lowest_gini_val = curr_gini_val
                    best_node = curr_node

            ##---Add New Node To Tree And Change Old Tree Elements---##
            for child_key, child_element in element.get_parent().get_children().items():
                if child_element == element:
                    ##---Re-Reference Changed Child To Parent---##
                    element.get_parent().get_children()[child_key] = best_node
                    ##---Add New Node To Tree---###
                    tree[key] = best_node

                    ##---Add New Node Children To Tree---##
                    for i in best_node.children.items():
                        add_to_tree(i[1])
                    break;
    return

### Training Tree
The train_tree method performs the tree's initial division, and continues to call the partition method to further purify and split the training data into their seperated leafs.

In [9]:
def train_tree(data, predictors, target):

    ##---Create Root Node---##
    lowest_gini_val = float('inf')
    best_node = None

    for predictor in predictors:
        curr_node = node(data, predictor, None, None)
        curr_node.partitioning()
        curr_gini_val = curr_node.calculate_total_gini_impurity(curr_node.get_children())
        
        if curr_gini_val < lowest_gini_val:
            lowest_gini_val = curr_gini_val
            best_node = curr_node

    add_to_tree(best_node)
    for _, element in best_node.children.items():
        add_to_tree(element)

    ##---Loop Partitioning Until Pure---##
    tree_is_impure = True
    
    while tree_is_impure:
        tree_gini_impurity = 0.0
        for _, element in tree.items():
            if not isinstance(element, leaf):
                continue;
            tree_gini_impurity += element.calculate_gini_impurity()
        if tree_gini_impurity == 0.0:
            tree_is_impure = False
        else:
            partition()

    ##---Add Output Values For Each Leaf---##
    for _, element in tree.items():
        if not isinstance(element, leaf):
            continue;
        element.set_output_value()

In [10]:
train_data = pd.DataFrame({"Smokes":[False, False, True, False, True, True, True, False, True, False, False],
                    "Drinks Alcohol":[False, False, True, True, False, False, False, True, True, False, False],
                    "Age":[7,12,18,19,21, 23, 28, 29, 32, 50, 81],
                    "Gender":[1, 2, 1, 2, 1, 2, 1, 1, 2, 1, 1], # 1 = Male, 2 = Female
                    "Has Chronic Disease":[False, False, False, False, True, True, False, True, True, False, True]})

train_data['Gender'] = train_data['Gender'].astype('category')
train_data['Has Chronic Disease'] = train_data['Has Chronic Disease'].astype('category')

In [11]:
tree = {}
predictors = train_data.columns[:-1]
target = train_data.columns[-1]

train_tree(train_data, predictors, target)

### Testing Tree
The test_tree is used to predict and assign the unseen data to their respectable leafs.

In [13]:
def test_tree(tree, test_data):

    ##---Loop Every Test Data Row---##
    for index, row in test_data.iterrows():
        ##---Remains False Until Curr Unseen Data Has Been Set An Output Value---##
        found_result = False
        ##---Keeps Track Of Curr Tree Testing Position, Always Begin At Partition 0---##
        curr_tree_pos = tree['partition_0']

        while found_result == False:
            ##---End Loop If Curr Tree Pos Is A Leaf---##
            if isinstance(curr_tree_pos, leaf):
                found_result = True
                test_data.loc[index, 'Output'] = curr_tree_pos.get_output_value()
            else:
                ##---Find Next Curr Tree Position---##
                children = curr_tree_pos.get_children().items()
                for _, child in children:
                    cond = child.get_branch_condition()
                    if cond(row) == True:
                        curr_tree_pos = child
                        break;
    return test_data

In [14]:
test_data = pd.DataFrame({"Smokes":[False, True],
                    "Drinks Alcohol":[False, True],
                    "Age":[9, 32],
                    "Gender":[1, 1]})

In [15]:
result = test_tree(tree, test_data)

In [16]:
result

,Smokes,Drinks Alcohol,Age,Gender,Output
0,False,False,9,1,False
1,True,True,32,1,True
